In [1]:
import pandas as pd
df = pd.read_csv('../../02_Data/processed/real_final_ml.csv')

c:\Users\seon\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\seon\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = df.drop(columns=['enableBoardGameProperties','projectID','campaignGoal_usd_6m','fundsGathered_usd_6m','price_usd_6m','is_backer_0','is_backer_1',"fundedInSeconds"])

In [3]:
import numpy as np
import pandas as pd
import warnings
import lightgbm as lgb
import re
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# ======================================================================
# 1. 1.5 IQR 기반 타깃 변수(fundsGathered_usd_1m) 아웃라이어 제거
# ======================================================================
target_col = 'fundsGathered_usd_1m'

q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# 정상 범위 내 프로젝트만 필터링 (> 0 조건 포함)
df_filtered = df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)].copy()
df_filtered = df_filtered[df_filtered[target_col] > 0].reset_index(drop=True)

print(f"🧹 1.5 IQR 타깃 정제 완료: {df_filtered.shape[0]}행 확보")

# ======================================================================
# 2. X_full (나머지 모든 변수) 및 y (타깃 로그 변환) 분리
# ======================================================================
X_full = df_filtered.drop(columns=[target_col])
y_original = df_filtered[target_col]
y_log = np.log1p(y_original)

# LightGBM 문자열 깨짐(특수문자/공백) 방지 안전장치
X_full.columns = [re.sub(r'[ ,\{\}:"\]\[\-]', '_', col) for col in X_full.columns]
print(f"🎒 X_full 매트릭스 빌드 완료: 총 {X_full.shape[1]}개의 독립변수 탑재")

# ======================================================================
# 3. 5-Fold 교차 검증 기반 최초의 Full Model 성능 측정
# ======================================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_tr_mae, cv_te_mae = [], []
cv_tr_r2, cv_te_r2 = [], []

# 전체 변수의 중요도(Gain 기준)를 누적할 배열 초기화
feature_importances = np.zeros(X_full.shape[1])

print("\n🏋️ 모든 변수를 투입한 Full LightGBM 모델 5-Fold 학습 시작...")

for train_idx, test_idx in kf.split(X_full):
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr_log, y_te_log = y_log.iloc[train_idx], y_log.iloc[test_idx]
    
    # 변수가 많으므로 과적합 방지를 위해 기본 규제 매개변수를 가볍게 세팅한 모델
    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X_tr, y_tr_log)
    
    # 예측 및 역변환(expm1)
    tr_preds = np.expm1(model.predict(X_tr))
    te_preds = np.expm1(model.predict(X_te))
    
    y_tr_real = y_original.iloc[train_idx]
    y_te_real = y_original.iloc[test_idx]
    
    # 지표 산출
    cv_tr_mae.append(mean_absolute_error(y_tr_real, tr_preds))
    cv_te_mae.append(mean_absolute_error(y_te_real, te_preds))
    cv_tr_r2.append(r2_score(y_tr_real, tr_preds))
    cv_te_r2.append(r2_score(y_te_real, te_preds))
    
    # 변수 중요도(기본값: Gain) 누적 (5개 폴드 평균값 계산용)
    feature_importances += model.booster_.feature_importance(importance_type='gain') / 5

# ======================================================================
# 4. 베이스라인 성적표 및 탈락 후보군(중요도 하위 20개) 출력
# ======================================================================
print("\n🏆 [1단계: Full Model 성적표 (기준점)]")
print("=" * 65)
print(f" 📊 1. 평균 절대 오차 (MAE)")
print(f"    - Train MAE : {np.mean(cv_tr_mae):,.2f} 달러")
print(f"    - Test MAE  : {np.mean(cv_te_mae):,.2f} 달러")
print(f" 📈 2. 결정계수 (R² Score)")
print(f"    - Train R²  : {np.mean(cv_tr_r2):.4f}")
print(f"    - Test R²   : {np.mean(cv_te_r2):.4f}")
print("=" * 65)

# 중요도 데이터프레임 빌드
df_imp = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n🚨 [후진제거 1순위] 변수 중요도 하위 20개 피처 목록")
print("-" * 65)
print(df_imp.tail(20))
print("-" * 65)

c:\Users\seon\anaconda3\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


🧹 1.5 IQR 타깃 정제 완료: 415행 확보
🎒 X_full 매트릭스 빌드 완료: 총 148개의 독립변수 탑재

🏋️ 모든 변수를 투입한 Full LightGBM 모델 5-Fold 학습 시작...

🏆 [1단계: Full Model 성적표 (기준점)]
 📊 1. 평균 절대 오차 (MAE)
    - Train MAE : 7,199.64 달러
    - Test MAE  : 29,334.98 달러
 📈 2. 결정계수 (R² Score)
    - Train R²  : 0.9818
    - Test R²   : 0.8064

🚨 [후진제거 1순위] 변수 중요도 하위 20개 피처 목록
-----------------------------------------------------------------
                Feature  Importance
128                Cats         0.0
129        Civilization         0.0
130  Collectible_Models         0.0
131       Deck_Building         0.0
132           Deduction         0.0
133           Dexterity         0.0
134  currencySymbol_CAD         0.0
135             Digital         0.0
136            Economic         0.0
137         Educational         0.0
138         Exploration         0.0
139  currencySymbol_AUD         0.0
140               Feast         0.0
141        First_Timers         0.0
142     Game_Components         0.0
143              Legacy   

In [4]:
# ======================================================================
# 🚀 2단계: 자동화 후진 제거법 (Backward Elimination Loop)
# ======================================================================

# 1. 기여도가 정확히 0.0인 변수들 1차로 일괄 식별 및 제거
zero_features = df_imp[df_imp['Importance'] == 0.0]['Feature'].tolist()
print(f"🗑️ 1차 청소: 기여도가 전혀 없는(Importance == 0) 변수 {len(zero_features)}개 즉시 탈락.")
print(f"   * 탈락 변수 일부 예시: {zero_features[:5]}")

# 0.0인 변수를 제외한 현재 정예 멤버로 변수 리스트 초기화
current_features = df_imp[df_imp['Importance'] > 0.0]['Feature'].tolist()
print(f"🏃 서바이벌 레이스 진입 변수: 총 {len(current_features)}개")

# 최적의 스코어를 기록하기 위한 추적용 변수들
best_mae = float('inf')
best_feature_set = []
history = []

step = 1

# 하위 변수를 하나씩 제거하며 성능 벼랑 끝 테스트 진행
while len(current_features) > 2: # 최소 2개 변수는 남을 때까지 반복
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_te_mae = []
    
    # 현재 남은 변수셋으로만 데이터 슬라이싱
    X_step = X_full[current_features]
    
    for train_idx, test_idx in kf.split(X_step):
        X_tr, X_te = X_step.iloc[train_idx], X_step.iloc[test_idx]
        y_tr_log = y_log.iloc[train_idx]
        
        model = lgb.LGBMRegressor(
            n_estimators=300, learning_rate=0.03, 
            random_state=42, n_jobs=-1, verbose=-1
        )
        model.fit(X_tr, y_tr_log)
        
        te_preds = np.expm1(model.predict(X_te))
        cv_te_mae.append(mean_absolute_error(y_original.iloc[test_idx], te_preds))
        
    current_mae = np.mean(cv_te_mae)
    
    # 중요도 재측정 (전체 데이터 기반)เพื่อ 파악하위 변수
    full_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    full_model.fit(X_step, y_log)
    
    # 현재 변수 세트 내에서의 기여도 정렬
    step_imp = pd.DataFrame({
        'Feature': current_features,
        'Importance': full_model.booster_.feature_importance(importance_type='gain')
    }).sort_values(by='Importance', ascending=True).reset_index(drop=True)
    
    # 기록 저장
    history.append({'num_features': len(current_features), 'mae': current_mae})
    
    # 최적의 성적 경신 시 락인(Lock-in)
    if current_mae < best_mae:
        best_mae = current_mae
        best_feature_set = list(current_features)
        
    # 출력 흐름 정돈
    if step % 10 == 1 or len(current_features) <= 15:
        print(f"📍 [Step {step:02d}] 현재 남은 변수: {len(current_features)}개 ➡️ Test MAE: {current_mae:,.2f} 달러")
        
    # 중요도 꼴찌 변수 1개 탈락 타겟팅
    lowest_feature = step_imp.iloc[0]['Feature']
    current_features.remove(lowest_feature)
    step += 1

print("\n" + "="*65)
print("🏆 [후진 제거법 서바이벌 최종 성적표]")
print("="*65)
print(f"🎯 역대 최저 실전 오차 (Best Test MAE): {best_mae:,.2f} 달러")
print(f"📦 모델이 선택한 최종 정예 변수 개수 : {len(best_feature_set)}개")
print(f"🗑️ 살을 도려내며 절감한 오차 규모    : {28462.56 - best_mae:,.2f} 달러 감소")
print("="*65)

# 나중을 위해 최종 정예 변수 리스트를 파일 형태로 백업
pd.Series(best_feature_set).to_csv('backward_selected_features.csv', index=False)

🗑️ 1차 청소: 기여도가 전혀 없는(Importance == 0) 변수 39개 즉시 탈락.
   * 탈락 변수 일부 예시: ['WinterFeast2026', 'OctoberFeast25', 'Paint', 'Worker_placement', 'Political']
🏃 서바이벌 레이스 진입 변수: 총 109개
📍 [Step 01] 현재 남은 변수: 109개 ➡️ Test MAE: 29,334.98 달러
📍 [Step 11] 현재 남은 변수: 99개 ➡️ Test MAE: 29,428.82 달러
📍 [Step 21] 현재 남은 변수: 89개 ➡️ Test MAE: 28,732.03 달러
📍 [Step 31] 현재 남은 변수: 79개 ➡️ Test MAE: 28,729.36 달러
📍 [Step 41] 현재 남은 변수: 69개 ➡️ Test MAE: 29,641.68 달러
📍 [Step 51] 현재 남은 변수: 59개 ➡️ Test MAE: 29,599.91 달러
📍 [Step 61] 현재 남은 변수: 49개 ➡️ Test MAE: 29,973.39 달러
📍 [Step 71] 현재 남은 변수: 39개 ➡️ Test MAE: 30,669.79 달러
📍 [Step 81] 현재 남은 변수: 29개 ➡️ Test MAE: 30,966.13 달러
📍 [Step 91] 현재 남은 변수: 19개 ➡️ Test MAE: 31,140.86 달러
📍 [Step 95] 현재 남은 변수: 15개 ➡️ Test MAE: 30,013.24 달러
📍 [Step 96] 현재 남은 변수: 14개 ➡️ Test MAE: 29,465.75 달러
📍 [Step 97] 현재 남은 변수: 13개 ➡️ Test MAE: 30,026.22 달러
📍 [Step 98] 현재 남은 변수: 12개 ➡️ Test MAE: 29,839.88 달러
📍 [Step 99] 현재 남은 변수: 11개 ➡️ Test MAE: 29,955.78 달러
📍 [Step 100] 현재 남은 변수: 10개 ➡️ Test MAE: 29,5

In [11]:
# ======================================================================
# 🔎 Step 3: 최종 살아남은 5대 정예 피처 명단 및 기여도 확인
# ======================================================================

# 백업된 CSV 파일에서 최종 정예 멤버 리스트 불러오기
final_features = pd.read_csv('backward_selected_features.csv')['0'].tolist()

print("👑 [최종 확정] LightGBM이 선택한 5대 정예 피처")
print("="*65)
for i, f in enumerate(final_features, 1):
    print(f" 🔥 {i}위 피처 : {f}")
print("="*65)

# 이 5개 변수만 가지고 전체 데이터 기준의 최종 기여도(Gain) 확인
X_final = X_full[final_features]
final_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
final_model.fit(X_final, y_log)

df_final_imp = pd.DataFrame({
    '정예 피처명': final_features,
    '최종 기여도(Gain)': final_model.booster_.feature_importance(importance_type='gain')
}).sort_values(by='최종 기여도(Gain)', ascending=False).reset_index(drop=True)

print("\n📊 5대 정예 피처 내부 서열(Gain) 및 수치")
print("-" * 65)
print(df_final_imp)
print("-" * 65)

👑 [최종 확정] LightGBM이 선택한 5대 정예 피처
 🔥 1위 피처 : is_pledge_master_1
 🔥 2위 피처 : Product_Question_count_1
 🔥 3위 피처 : fundedInSeconds
 🔥 4위 피처 : likes_1
 🔥 5위 피처 : campaignGoal_usd_1m

📊 5대 정예 피처 내부 서열(Gain) 및 수치
-----------------------------------------------------------------
                     정예 피처명  최종 기여도(Gain)
0        is_pledge_master_1  12727.763664
1  Product_Question_count_1  11270.887420
2           fundedInSeconds   5056.747562
3       campaignGoal_usd_1m   4141.495013
4                   likes_1   3069.004011
-----------------------------------------------------------------


In [13]:
import numpy as np
import pandas as pd
import warnings
import re
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# ======================================================================
# [단계 1] 데이터 로드 및 맞춤형 변수 드랍 (Data Cleaning)
# ======================================================================
print("🔄 1단계: 데이터 로드 및 초기 정제 시작...")
df_raw = pd.read_csv('../../02_Data/processed/real_final_ml.csv')

# 유출(Leakage) 리스크 변수, 6개월 타깃, 텍스트 원문(projectTags) 일괄 제거
drop_columns = [
    'enableBoardGameProperties', 'projectID', 'backersCount', 
    'phaseLabel', 'isFeatured', 'installmentMinPayment', 
    'hasLimitedStock', 'productCanBePurchased', 
    'campaignGoal_usd_6m', 'fundsGathered_usd_6m', 'price_usd_6m',
    'is_backer_0', 'is_backer_1', 'projectTags',"fundedInSeconds"
]
df_cleaned = df_raw.drop(columns=drop_columns, errors='ignore')
print(f"   - 원본 데이터 크기: {df_raw.shape[0]}행, {df_raw.shape[1]}열")
print(f"   - 불필요 변수 {len(drop_columns)}개 제거 완료 ➡️ 현재 열: {df_cleaned.shape[1]}개")

# ======================================================================
# [단계 2] 통계학적 전처리: 1.5 IQR 기반 타깃 변수 아웃라이어 제거
# ======================================================================
target_col = 'fundsGathered_usd_1m'

q1 = df_cleaned[target_col].quantile(0.25)
q3 = df_cleaned[target_col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# 정상 범위 프로젝트 및 펀딩 성공(>0) 데이터 필터링
df_filtered = df_cleaned[(df_cleaned[target_col] >= lower_bound) & (df_cleaned[target_col] <= upper_bound)].copy()
df_filtered = df_filtered[df_filtered[target_col] > 0].reset_index(drop=True)
print(f"🧹 2단계: 1.5 IQR 타깃 정제 완료 (최종 학습 데이터 행 크기: {df_filtered.shape[0]}행)")

# ======================================================================
# [단계 3] X (독립변수) / y (타깃 로그 변환) 분리 및 안전장치
# ======================================================================
X_full = df_filtered.drop(columns=[target_col])
y_original = df_filtered[target_col]
y_log = np.log1p(y_original)

# LightGBM/XGBoost 특수문자 및 공백 에러 방지 정규화
X_full.columns = [re.sub(r'[ ,\{\}:"\]\[\-]', '_', col) for col in X_full.columns]
print(f"🎒 3단계: 독립변수 Matrix 구축 완료 (총 {X_full.shape[1]}개 피처 대기 중)")

# ======================================================================
# [단계 4] Full Model 최초 5-Fold CV 성능 측정 및 초기 변수 중요도 획득
# ======================================================================
print("\n🏋️ 4단계: 130여 개 전체 변수 투입 최초 Full Model 5-Fold 학습 시작...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

base_tr_mae, base_te_mae = [], []
base_tr_r2, base_te_r2 = [], []
feature_importances = np.zeros(X_full.shape[1])

for train_idx, test_idx in kf.split(X_full):
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr_log = y_log.iloc[train_idx]
    
    model = LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_tr, y_tr_log)
    
    tr_preds = np.expm1(model.predict(X_tr))
    te_preds = np.expm1(model.predict(X_te))
    
    base_tr_mae.append(mean_absolute_error(y_original.iloc[train_idx], tr_preds))
    base_te_mae.append(mean_absolute_error(y_original.iloc[test_idx], te_preds))
    base_tr_r2.append(r2_score(y_original.iloc[train_idx], tr_preds))
    base_te_r2.append(r2_score(y_original.iloc[test_idx], te_preds))
    
    # 기여도(Gain 기준) 누적
    feature_importances += model.booster_.feature_importance(importance_type='gain') / 5

print("\n📊 [기준점] 모든 변수를 다 넣었을 때의 첫 성적표")
print("-" * 70)
print(f"   • Train MAE : {np.mean(base_tr_mae):,.2f}$ | Test MAE  : {np.mean(base_te_mae):,.2f}$")
print(f"   • Train R²  : {np.mean(base_tr_r2):.4f}  | Test R²   : {np.mean(base_te_r2):.4f}")
print("-" * 70)

# 마스터 중요도 데이터프레임 빌드
df_imp = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# ======================================================================
# [단계 5] 업그레이드 다중 지표 모니터링 후진 제거법 루프 가동
# ======================================================================
# 기여도 0.0 변수 일괄 식별 및 제거
zero_features = df_imp[df_imp['Importance'] == 0.0]['Feature'].tolist()
print(f"\n🗑️ 5단계: 기여도가 전혀 없는(Importance == 0) 변수 {len(zero_features)}개 1차 일괄 청소.")

current_features = df_imp[df_imp['Importance'] > 0.0]['Feature'].tolist()
print(f"🏃 서바이벌 레이스 최종 진입 변수: 총 {len(current_features)}개\n" + "="*70)

best_mae = float('inf')
best_feature_set = []
best_full_metrics = {}
history = []

step = 1

# 변수가 2개 남을 때까지 꼴찌 변수를 하나씩 치우며 벼랑 끝 레이스 진행
while len(current_features) > 2:
    fold_tr_mae, fold_te_mae = [], []
    fold_tr_r2, fold_te_r2 = [], []
    fold_te_medae, fold_te_rmse = [], []
    
    X_step = X_full[current_features]
    
    for train_idx, test_idx in kf.split(X_step):
        X_tr, X_te = X_step.iloc[train_idx], X_step.iloc[test_idx]
        y_tr_log = y_log.iloc[train_idx]
        
        loop_model = LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
        loop_model.fit(X_tr, y_tr_log)
        
        tr_preds = np.expm1(loop_model.predict(X_tr))
        te_preds = np.expm1(loop_model.predict(X_te))
        
        y_tr_real = y_original.iloc[train_idx]
        y_te_real = y_original.iloc[test_idx]
        
        fold_tr_mae.append(mean_absolute_error(y_tr_real, tr_preds))
        fold_te_mae.append(mean_absolute_error(y_te_real, te_preds))
        fold_tr_r2.append(r2_score(y_tr_real, tr_preds))
        fold_te_r2.append(r2_score(y_te_real, te_preds))
        fold_te_medae.append(median_absolute_error(y_te_real, te_preds))
        fold_te_rmse.append(np.sqrt(mean_squared_error(y_te_real, te_preds)))
        
    step_metrics = {
        'num_features': len(current_features),
        'tr_mae': np.mean(fold_tr_mae),
        'te_mae': np.mean(fold_te_mae),
        'tr_r2': np.mean(fold_tr_r2),
        'te_r2': np.mean(fold_te_r2),
        'te_medae': np.mean(fold_te_medae),
        'te_rmse': np.mean(fold_te_rmse)
    }
    
    history.append(step_metrics)
    
    # 실전 오차(Test MAE) 기준 최고 기록 경신 시 변수셋 및 지표 통째로 Lock-in
    if step_metrics['te_mae'] < best_mae:
        best_mae = step_metrics['te_mae']
        best_feature_set = list(current_features)
        best_full_metrics = step_metrics
        
    # 간격 조절 및 마지노선 구간(15개 이하) 상세 리포트 출력
    if step % 10 == 1 or len(current_features) <= 15:
        print(f"📍 [Step {step:02d}] 현재 남은 변수: {step_metrics['num_features']}개")
        print(f"   - [MAE]  Train: {step_metrics['tr_mae']:,.1f}$ | Test: {step_metrics['te_mae']:,.1f}$ (🚨 갭: {step_metrics['te_mae'] - step_metrics['tr_mae']:,.1f}$)")
        print(f"   - [R²]   Train: {step_metrics['tr_r2']:.4f}  | Test: {step_metrics['te_r2']:.4f}")
        print(f"   - [보조] Test MedAE: {step_metrics['te_medae']:,.1f}$ | Test RMSE: {step_metrics['te_rmse']:,.1f}$")
        print("-" * 70)
        
    # 다음 탈락자 선정을 위한 중요도 재측정 (전체 데이터 기반)
    full_model = LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    full_model.fit(X_step, y_log)
    
    step_imp = pd.DataFrame({
        'Feature': current_features,
        'Importance': full_model.booster_.feature_importance(importance_type='gain')
    }).sort_values(by='Importance', ascending=True).reset_index(drop=True)
    
    # 기여도 최하위 변수 1개 탈락 처리
    lowest_feature = step_imp.iloc[0]['Feature']
    current_features.remove(lowest_feature)
    step += 1

# ======================================================================
# [단계 6] 최종 서바이벌 우승 성적표 마스터 박제
# ======================================================================
print("\n" + "="*70)
print("👑 [후진 제거법 종합 평가지표 마스터 성적표]")
print("="*70)
print(f"🎯 1. 최적의 정예 변수 개수 : {best_full_metrics['num_features']}개 최종 낙점")
print(f"📊 2. 평균 절대 오차 (MAE)")
print(f"   - Train MAE : {best_full_metrics['tr_mae']:,.2f} 달러")
print(f"   - Test MAE  : {best_full_metrics['te_mae']:,.2f} 달러 (역대 최저 오차 수립)")
print(f"📈 3. 결정계수 (R² Score)")
print(f"   - Train R²  : {best_full_metrics['tr_r2']:.4f}")
print(f"   - Test R²   : {best_full_metrics['te_r2']:.4f} (실전 데이터 설명력)")
print(f"🎯 4. 중간값 절대 오차 (MedAE) : {best_full_metrics['te_medae']:,.2f} 달러")
print(f"🛡️ 5. 평균 제곱근 오차 (RMSE)  : {best_full_metrics['te_rmse']:,.2f} 달러")
print("="*70)

# 훗날을 위해 최종 정예 피처 목록 저장
pd.Series(best_feature_set).to_csv('backward_selected_features.csv', index=False)
print("💾 최적의 정예 피처 목록 리스트가 'backward_selected_features.csv' 파일로 안전하게 백업되었습니다.")

🔄 1단계: 데이터 로드 및 초기 정제 시작...
   - 원본 데이터 크기: 483행, 157열
   - 불필요 변수 15개 제거 완료 ➡️ 현재 열: 149개
🧹 2단계: 1.5 IQR 타깃 정제 완료 (최종 학습 데이터 행 크기: 415행)
🎒 3단계: 독립변수 Matrix 구축 완료 (총 148개 피처 대기 중)

🏋️ 4단계: 130여 개 전체 변수 투입 최초 Full Model 5-Fold 학습 시작...

📊 [기준점] 모든 변수를 다 넣었을 때의 첫 성적표
----------------------------------------------------------------------
   • Train MAE : 7,199.64$ | Test MAE  : 29,334.98$
   • Train R²  : 0.9818  | Test R²   : 0.8064
----------------------------------------------------------------------

🗑️ 5단계: 기여도가 전혀 없는(Importance == 0) 변수 39개 1차 일괄 청소.
🏃 서바이벌 레이스 최종 진입 변수: 총 109개
📍 [Step 01] 현재 남은 변수: 109개
   - [MAE]  Train: 7,199.6$ | Test: 29,335.0$ (🚨 갭: 22,135.3$)
   - [R²]   Train: 0.9818  | Test: 0.8064
   - [보조] Test MedAE: 8,234.4$ | Test RMSE: 57,115.0$
----------------------------------------------------------------------
📍 [Step 11] 현재 남은 변수: 99개
   - [MAE]  Train: 7,109.2$ | Test: 29,428.8$ (🚨 갭: 22,319.6$)
   - [R²]   Train: 0.9819  | Test: 0.8055
   - [보조] Test MedAE: 7,

In [14]:
final_5_features = pd.read_csv('backward_selected_features.csv')['0'].tolist()

print("👑 [최종 생존] 140여 개 경쟁자를 뚫고 살아남은 5대 피처")
print("=" * 65)
for i, name in enumerate(final_5_features, 1):
    print(f" 🔥 {i}위 피처 : {name}")
print("=" * 65)

👑 [최종 생존] 140여 개 경쟁자를 뚫고 살아남은 5대 피처
 🔥 1위 피처 : is_pledge_master_1
 🔥 2위 피처 : 중립_1
 🔥 3위 피처 : Product_Question_count_1
 🔥 4위 피처 : softclose
 🔥 5위 피처 : likes_1
 🔥 6위 피처 : campaignGoal_usd_1m


In [17]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# 1. 최적의 5대 정예 피처만 슬라이싱
final_5_features = ['is_pledge_master_1', '중립_1','Product_Question_count_1', 'softclose', 'likes_1', 'campaignGoal_usd_1m']
X_v5 = X_full[final_5_features]

print(f"🎯 독수리 오형제 피처만 들고 초정밀 Optuna 튜닝 스타트! (Matrix 크기: {X_v5.shape})")

def objective(trial):
    # LightGBM의 핵심 하이퍼파라미터 탐색 범위 지정
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 7, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 30),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_maes = []
    
    for train_idx, test_idx in kf.split(X_v5):
        X_tr, X_te = X_v5.iloc[train_idx], X_v5.iloc[test_idx]
        y_tr_log = y_log.iloc[train_idx]
        
        opt_model = lgb.LGBMRegressor(**params)
        opt_model.fit(X_tr, y_tr_log)
        
        preds = np.expm1(opt_model.predict(X_te))
        cv_maes.append(mean_absolute_error(y_original.iloc[test_idx], preds))
        
    return np.mean(cv_maes)

# Optuna 최적화 세션 실행 (시간 관계상 깔끔하게 100회만 타격)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print("\n" + "="*70)
print("👑 [Optuna 초정밀 튜닝 완료 성적표]")
print("="*70)
print(f"🥇 5대 피처 기반 역대 최저 오차 (Best CV MAE): {study.best_value:,.2f} 달러")
print("\n🛠️ 우승 파라미터 조합:")
for key, value in study.best_params.items():
    print(f"   • {key}: {value}")
print("="*70)

# 최종 튜닝 파라미터 저장
import json
with open('lgbm_best_params_v5.json', 'w') as f:
    json.dump(study.best_params, f, indent=4)

🎯 독수리 오형제 피처만 들고 초정밀 Optuna 튜닝 스타트! (Matrix 크기: (415, 6))


  0%|          | 0/100 [00:00<?, ?it/s]


👑 [Optuna 초정밀 튜닝 완료 성적표]
🥇 5대 피처 기반 역대 최저 오차 (Best CV MAE): 25,893.50 달러

🛠️ 우승 파라미터 조합:
   • n_estimators: 950
   • learning_rate: 0.012313916334664792
   • num_leaves: 13
   • max_depth: 3
   • min_child_samples: 17
   • subsample: 0.8175679679177557
   • colsample_bytree: 0.6822100693509103
   • reg_alpha: 0.0028729941781964895
   • reg_lambda: 0.007735628570598627


In [20]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score

# 1. 5대 정예 피처 및 Optuna 우승 파라미터 동기화
final_5_features = ['is_pledge_master_1', '중립_1','Product_Question_count_1', 'softclose', 'likes_1', 'campaignGoal_usd_1m']
X_v5 = X_full[final_5_features]

# 방금 획득한 따끈따끈한 베스트 파라미터 로드
best_tuned_params = {
    'n_estimators': 950,
    'learning_rate': 0.012313916334664792,
    'num_leaves': 13,
    'max_depth': 3,
    'min_child_samples': 17,
    'subsample': 0.8175679679177557,
    'colsample_bytree': 0.6822100693509103,
    'reg_alpha': 0.0028729941781964895,
    'reg_lambda': 0.007735628570598627,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

# 2. 5-Fold 교차 검증 검사대 가동
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_tr_mae, fold_te_mae = [], []
fold_tr_r2, fold_te_r2 = [], []
fold_te_medae, fold_te_rmse = [], []

print("🏋️ Optuna 우승 조합 기반 6대 마스터 지표 산출 중...")

for train_idx, test_idx in kf.split(X_v5):
    X_tr, X_te = X_v5.iloc[train_idx], X_v5.iloc[test_idx]
    y_tr_log = y_log.iloc[train_idx]
    
    # 최적화된 모델 학습
    model = lgb.LGBMRegressor(**best_tuned_params)
    model.fit(X_tr, y_tr_log)
    
    # 예측 및 역변환
    tr_preds = np.expm1(model.predict(X_tr))
    te_preds = np.expm1(model.predict(X_te))
    
    y_tr_real = y_original.iloc[train_idx]
    y_te_real = y_original.iloc[test_idx]
    
    # 지표 연산
    fold_tr_mae.append(mean_absolute_error(y_tr_real, tr_preds))
    fold_te_mae.append(mean_absolute_error(y_te_real, te_preds))
    fold_tr_r2.append(r2_score(y_tr_real, tr_preds))
    fold_te_r2.append(r2_score(y_te_real, te_preds))
    fold_te_medae.append(median_absolute_error(y_te_real, te_preds))
    fold_te_rmse.append(np.sqrt(mean_squared_error(y_te_real, te_preds)))

# 3. 최종 종합 성적표 디스플레이
print("\n" + "="*70)
print("👑 [최종 피날레: Optuna 튜닝 완료 종합 성적표]")
print("="*70)
print(f"🎯 1. 사용된 피처 개수 : 5개 (독수리 오형제 락인)")
print(f"📊 2. 평균 절대 오차 (MAE)")
print(f"   - Train MAE : {np.mean(fold_tr_mae):,.2f} 달러")
print(f"   - Test MAE  : {np.mean(fold_te_mae):,.2f} 달러 (🚨 갭: {np.mean(fold_te_mae) - np.mean(fold_tr_mae):,.2f} 달러)")
print(f"📈 3. 결정계수 (R² Score)")
print(f"   - Train R²  : {np.mean(fold_tr_r2):.4f}")
print(f"   - Test R²   : {np.mean(fold_te_r2):.4f} (설명력)")
print(f"🎯 4. 중간값 절대 오차 (MedAE) : {np.mean(fold_te_medae):,.2f} 달러")
print(f"🛡️ 5. 평균 제곱근 오차 (RMSE)  : {np.mean(fold_te_rmse):,.2f} 달러")
print("="*70)

🏋️ Optuna 우승 조합 기반 6대 마스터 지표 산출 중...

👑 [최종 피날레: Optuna 튜닝 완료 종합 성적표]
🎯 1. 사용된 피처 개수 : 5개 (독수리 오형제 락인)
📊 2. 평균 절대 오차 (MAE)
   - Train MAE : 21,222.13 달러
   - Test MAE  : 25,893.50 달러 (🚨 갭: 4,671.37 달러)
📈 3. 결정계수 (R² Score)
   - Train R²  : 0.8956
   - Test R²   : 0.8489 (설명력)
🎯 4. 중간값 절대 오차 (MedAE) : 6,206.01 달러
🛡️ 5. 평균 제곱근 오차 (RMSE)  : 50,288.95 달러
